<a href="https://colab.research.google.com/github/k9Sx3CC/01_first_look_and_discovery.ipynb/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k9Sx3CC/flyrank-internship-test/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1

The FlyRank research reports that machine learning can prioritize declining content more effectively than a rule-based refresh strategy.

The label comes from historical content performance, where pages are labeled as declining based on their observed trend. This is a reasonable target because it is derived from measured historical data. However, the strength of the claim depends on the validation design. A grouped client_holdout split provides stronger evidence because it evaluates the model on completely unseen clients rather than allowing similar pages from the same client to appear in both training and testing.

Finding 2

The paper also reports that combining multiple search and engagement features improves refresh recommendations.

The label again comes from historical observations of page performance. This claim is supported only if all features are available before prediction and no future information is included. An honest validation design should therefore remove leakage features and evaluate the model on unseen clients or future data so the reported performance reflects realistic deployment.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week 5 model already used the client_holdout split strategy, which groups pages by client. This prevents information from the same client appearing in both training and testing and provides a more realistic estimate of generalization.

The measured results were:


*   Model: Week 4 Baseline, Precision@50: 0.24
*   Model: Random Forest (client_holdout), Precision@50: 0.74

The Random Forest achieved a substantially higher Precision@50 than the rule-based baseline under the same grouped evaluation. This observed improvement suggests the model is more effective at prioritizing declining pages in this dataset, although additional validation on future data would strengthen the claim.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json
import pandas as pd
import os
import subprocess
import sys
# Set the repository path
REPO_DIR = "/content/flyrank-ml-internship-starter" # Clone only if it doesn't already exist
if not os.path.exists(REPO_DIR): subprocess.run([ "git", "clone", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR ], check=True) # Always go to the absolute repository path
os.chdir(REPO_DIR) # Add current repo to Python's search path
sys.path.append(os.getcwd())
print("Current directory:", os.getcwd())
print("Contents:", os.listdir())
print(os.listdir("scripts"))
!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py
!python scripts/03_train_model.py
!python scripts/04_evaluate_and_export.py
with open("outputs/model_results.json") as f:
  results = json.load(f)
  print(json.dumps(results, indent=2))
comparison = pd.DataFrame({
    "Method": [
        "Baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Precision@50": [
        results["baseline"]["baseline_precision_at_50"],
        results["models"]["logistic_regression"]["precision_at_50"],
        results["models"]["decision_tree"]["precision_at_50"],
        results["models"]["random_forest"]["precision_at_50"],
    ]
})

comparison


Current directory: /content/flyrank-ml-internship-starter
Contents: ['SETUP.md', 'LICENSE', 'README.md', 'DATA_USE.md', 'work', '.git', 'submission', 'requirements.txt', '.gitignore', 'scripts', 'docs', 'GUIDE.md', 'data', 'skills', 'notebooks', 'CLAUDE.md', 'outputs', 'AGENTS.md', '.github']
['ml_utils.py', '04_evaluate_and_export.py', '05_build_pdf_report.py', 'run_all.py', '03_train_model.py', '02_baseline_score.py', '01_prepare_features.py', '__pycache__']
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-

,Method,Precision@50
0,Baseline,0.24
1,Logistic Regression,0.40
2,Decision Tree,0.58
3,Random Forest,0.74


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I reviewed the final feature set for possible target leakage.

The target variable (is_declining_label) was created from trend_direction, but trend_direction itself was not included as an input feature.

During earlier experiments, trend_pct was identified as a potential leakage feature because it is closely related to the target and produced unrealistically high performance when included. It was therefore not used as a predictor in the final model. The remaining features describe historical visibility, engagement, freshness, and content characteristics that would be available before prediction. Based on this audit, I did not identify obvious target leakage in the final feature set.

The remaining features describe historical search visibility, engagement, freshness, and content characteristics that would be available before making a prediction. Based on this audit, the final model appears to avoid obvious target leakage.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

importance = pd.DataFrame(
    results["best_model"]["feature_importance_top"]
)

importance.head(10)

,feature,importance
0,days_with_impressions,0.158144
1,log_impressions_90d,0.128638
2,avg_position,0.109164
3,content_age_days,0.095168
4,char_count,0.042608
5,word_count,0.039609
6,log_clicks_90d,0.034463
7,ctr,0.033295
8,scroll_rate,0.031226
9,days_with_sessions,0.027995


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In this dataset, the Random Forest achieved the highest measured Precision@50 under a client-holdout evaluation. These results suggest that the model can support prioritizing pages for manual review, but additional validation on future data would be needed before using it for automated content refresh decisions.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.